# Simple recursive network for sequence learning

In [1]:
import sys
from pathlib import Path
import numpy as np
from numpy.typing import ArrayLike
import torch
from typing import Callable, Sequence
from tqdm import trange


# import local packages
sys.path.append(str(Path().resolve().parents[0]))

from src.utils.logger import setup_logger, add_log_level
from src.utils.args import get_args
from src.data.formater import PandasLoader


In [2]:
args = get_args()
logger_level = "TRACE"


In [3]:
add_log_level("TRACE", 5)
logger = setup_logger(name="logger", level=logger_level)
logger.info("logger set to info level")
logger.trace("TRACE logger activated")



INFO - logger set to info level
TRACE - TRACE logger activated


In [4]:
loader = PandasLoader(args["input"], args["format"])
responses = loader.get()
logger.info(f"subjects response per trial: \n {responses}")

# converting to array of shape (subjects, responses)
responses = responses.to_numpy().T
logger.debug(f"responses after transposition= {responses}")

# get set of responses values and their order of appearance(giving them a numerical id) restoring the value from it's encoded index is easy.
unique_vals, encoded = np.unique(responses.reshape(-1), return_inverse=True)
logger.info(f"unique values (set)={unique_vals}")
logger.info(f"encoded values={encoded}")
encoded = encoded.reshape(responses.shape)
logger.debug(f"final encoded shape (subjests, responses)={encoded.shape}")



INFO - subjects response per trial: 
 Subject    1  2  3  4  5  6  7  8  9  10  ... 51 52 53 54 55 56 57 58 59 60
TrialIndex                                ...                              
0           a  a  a  b  c  a  b  c  a  a  ...  a  a  a  b  c  a  b  c  b  b
1           J  Q  P  M  T  K  I  S  J  Q  ...  J  Q  P  M  T  K  I  S  L  H
2           d  d  d  e  f  d  e  f  d  d  ...  d  d  d  e  f  d  e  f  e  e
3           a  b  b  c  c  b  c  b  a  b  ...  a  b  b  c  c  b  c  b  b  c
4           S  I  O  Y  I  £  Q  H  S  I  ...  S  I  O  Y  I  £  Q  H  K  K
...        .. .. .. .. .. .. .. .. .. ..  ... .. .. .. .. .. .. .. .. .. ..
643         P  N  H  Y  £  G  H  M  P  N  ...  P  N  H  Y  £  G  H  M  H  W
644         d  e  d  e  d  d  f  f  d  e  ...  d  e  d  e  d  d  f  f  f  f
645         c  c  c  c  a  a  c  b  c  c  ...  c  c  c  c  a  a  c  b  b  c
646         O  I  K  ù  I  ù  &  Z  O  I  ...  O  I  K  ù  I  ù  &  Z  N  H
647         f  f  f  f  d  d  f  e  f  f  ...  f  

In [5]:
def make_uniform_tensor(
    extremum: tuple[float, float], shape: Sequence[int], grad: bool
):
    t = torch.empty(*shape)
    t.uniform_(*extremum)
    if grad:
        t.requires_grad_()
    return t


![test](https://web.stanford.edu/group/pdplab/pdphandbook/srn_net.png)

A SRN is a simplified RNN. The output of the hidden layer is fed back as input to the hidden layer at the
next time step. The output of the hidden layer is also used to compute the output of the network.

![SRN basic architecture](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.researchgate.net%2Fpublication%2F361380872%2Ffigure%2Ffig1%2FAS%3A1169080920875058%401655742020827%2FSchematic-diagram-of-Elman-network-structure-in-simple-recurrent-neural-network.jpg&f=1&nofb=1&ipt=1d5a36f3ef48ba69e88b9f96a9176f2e1ed8232b2019b9d3f7f7335e1ee85f1d)
```mermaid
graph TB;
hidden --> context
input --> hidden
context --> hidden
hidden --> output
```

In [6]:
class SRN_subject:
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        activation: list[Callable] = [torch.nn.Tanh],
        lr: float = 0.05,
        initial_w_unif: tuple[float, float] = (-0.1, 0.1),
        loss_fn: Callable = torch.nn.MSELoss()
    ):

        self.lr = lr
        self.activation = activation
        self.loss_fn = loss_fn

        self.Wxh = make_uniform_tensor(
            extremum=initial_w_unif, shape=[hidden_size, input_size + hidden_size], grad=True
        )
        
        self.Why = make_uniform_tensor(
            extremum=initial_w_unif, shape=[output_size, hidden_size], grad=True
        )

        self.context = torch.zeros(hidden_size)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None):
        # input of hidden layer: input concatenated with previous output of the hidden layer
        x_cat_context = torch.cat([x, self.context], dim=0)

        h = activation[0](self.Wxh @ x_cat_context)
        self.context = h.detach()  # detach the context from the computational graph to prevent backprop through time
        y = activation[1](self.Why @ h)
        return y

    def backprop(self, y_pred, y):


        loss = self.loss_fn(y_pred, y)
        loss.backward()

        if self.Wxh.grad is None or self.Why.grad is None:
            e = RuntimeError("gradient missing")
            logger.error(e)
            raise e

        with torch.no_grad():
            self.Wxh -= self.lr * self.Wxh.grad
            self.Why -= self.lr * self.Why.grad
            self.Wxh.grad.zero_()
            self.Why.grad.zero_()

        return loss




Question:
For the backpropagation, when we learn patern on on a screen, how do we see the cases with no stimulus? making them -1 (opposite to the cell with stimulus 1).

### hyperparameters

In [7]:

hidden_size = len(unique_vals)
activation = [torch.nn.Tanh (), torch.nn.Sigmoid()]
lr = 0.1
initial_w_unif = (-0.1, 0.1)
loss_fn = torch.nn.MSELoss()

extremum_grid = (0.,1.)


## Prediction and backprpagation

In [ ]:
input_size = len(unique_vals)
output_size = len(unique_vals)

for i in trange(encoded.shape[0]):
    srn_subject = SRN_subject(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size,
        activation=activation,
        lr=lr,
        initial_w_unif=initial_w_unif,
        loss_fn=loss_fn
    )

    for j in range(encoded.shape[1] - 1):
        x = encoded[i, j]
        y_true = encoded[i, j + 1]

        # make a grid with the previous value at the encoded label position.
        x_grid = torch.ones(len(unique_vals)) * extremum_grid[0]
        x_grid[x] = extremum_grid[1]

        # make a grid with the response value at the encoded label position.
        y_grid = torch.ones(len(unique_vals)) * extremum_grid[0]
        y_grid[y_true] = extremum_grid[1]

        y_pred_grid = srn_subject.forward(x_grid)
        loss = srn_subject.backprop(y_pred_grid, y_grid)


        y_true = unique_vals[y_true]
        y_pred = torch.argmax(y_pred_grid)
        y_pred_label = unique_vals[y_pred]
        x_label = unique_vals[x]
        logger.trace(
            f"subject {i}, trial {j}, x={x_label}, y={y_true}, y_pred={y_pred_label}, loss={loss}"
        )

  0%|                                                                                                            | 0/60 [00:00<?, ?it/s]

DEBUG - subject 0, trial 0, x=a, y=J, y_pred=R, loss=0.2503303289413452
DEBUG - subject 0, trial 1, x=J, y=d, y_pred=P, loss=0.25127139687538147
DEBUG - subject 0, trial 2, x=d, y=a, y_pred=L, loss=0.2501690983772278
DEBUG - subject 0, trial 3, x=a, y=S, y_pred=R, loss=0.2503645718097687
DEBUG - subject 0, trial 4, x=S, y=d, y_pred=U, loss=0.2504315674304962
DEBUG - subject 0, trial 5, x=d, y=a, y_pred=L, loss=0.2502674460411072
DEBUG - subject 0, trial 6, x=a, y=I, y_pred=R, loss=0.24995127320289612
DEBUG - subject 0, trial 7, x=I, y=d, y_pred=U, loss=0.25003746151924133
DEBUG - subject 0, trial 8, x=d, y=c, y_pred=L, loss=0.24997259676456451
DEBUG - subject 0, trial 9, x=c, y=M, y_pred=e, loss=0.24996842443943024
DEBUG - subject 0, trial 10, x=M, y=f, y_pred=X, loss=0.2498374879360199
DEBUG - subject 0, trial 11, x=f, y=a, y_pred=Q, loss=0.2493874728679657
DEBUG - subject 0, trial 12, x=a, y=&, y_pred=T, loss=0.24988417327404022
DEBUG - subject 0, trial 13, x=&, y=d, y_pred=b, loss=0

  2%|█▋                                                                                                  | 1/60 [00:01<01:40,  1.70s/it]

DEBUG - subject 1, trial 0, x=a, y=Q, y_pred=£, loss=0.24970455467700958
DEBUG - subject 1, trial 1, x=Q, y=d, y_pred=&, loss=0.2498706877231598
DEBUG - subject 1, trial 2, x=d, y=b, y_pred=M, loss=0.25114354491233826
DEBUG - subject 1, trial 3, x=b, y=I, y_pred=Y, loss=0.2509100139141083
DEBUG - subject 1, trial 4, x=I, y=e, y_pred=f, loss=0.25015002489089966
DEBUG - subject 1, trial 5, x=e, y=c, y_pred=U, loss=0.24952931702136993
DEBUG - subject 1, trial 6, x=c, y=J, y_pred=W, loss=0.24984939396381378
DEBUG - subject 1, trial 7, x=J, y=f, y_pred=M, loss=0.25066420435905457
DEBUG - subject 1, trial 8, x=f, y=a, y_pred=I, loss=0.24919432401657104
DEBUG - subject 1, trial 9, x=a, y=0, y_pred=£, loss=0.25015899538993835
DEBUG - subject 1, trial 10, x=0, y=d, y_pred=J, loss=0.2504499554634094
DEBUG - subject 1, trial 11, x=d, y=b, y_pred=K, loss=0.25060442090034485
DEBUG - subject 1, trial 12, x=b, y=X, y_pred=Y, loss=0.250365674495697
DEBUG - subject 1, trial 13, x=X, y=e, y_pred=T, loss

  3%|███▎                                                                                                | 2/60 [00:04<02:00,  2.08s/it]

...

DEBUG - subject 51, trial 0, x=a, y=Q, y_pred=I, loss=0.25025683641433716
DEBUG - subject 51, trial 1, x=Q, y=d, y_pred=£, loss=0.24998421967029572
DEBUG - subject 51, trial 2, x=d, y=b, y_pred=d, loss=0.2487078160047531
DEBUG - subject 51, trial 3, x=b, y=I, y_pred=ù, loss=0.25023216009140015
DEBUG - subject 51, trial 4, x=I, y=e, y_pred=f, loss=0.2503814101219177
DEBUG - subject 51, trial 5, x=e, y=c, y_pred=Y, loss=0.24877138435840607
DEBUG - subject 51, trial 6, x=c, y=J, y_pred=0, loss=0.24937745928764343
DEBUG - subject 51, trial 7, x=J, y=f, y_pred=G, loss=0.2508593201637268
DEBUG - subject 51, trial 8, x=f, y=a, y_pred=L, loss=0.2511560320854187
DEBUG - subject 51, trial 9, x=a, y=0, y_pred=I, loss=0.2506785988807678
DEBUG - subject 51, trial 10, x=0, y=d, y_pred=R, loss=0.25067612528800964
DEBUG - subject 51, trial 11, x=d, y=b, y_pred=d, loss=0.2493640035390854
DEBUG - subject 51, trial 12, x=b, y=X, y_pred=ù, loss=0.2509227395057678
DEBUG - subject 51, trial 13, x=X, y=e, y_

 87%|█████████████████████████████████████████████████████████████████████████████████████▊             | 52/60 [01:37<00:14,  1.79s/it]

DEBUG - subject 52, trial 0, x=a, y=P, y_pred=W, loss=0.24888022243976593
DEBUG - subject 52, trial 1, x=P, y=d, y_pred=0, loss=0.2494334578514099
DEBUG - subject 52, trial 2, x=d, y=b, y_pred=b, loss=0.2506037950515747
DEBUG - subject 52, trial 3, x=b, y=O, y_pred=N, loss=0.25207841396331787
DEBUG - subject 52, trial 4, x=O, y=e, y_pred=a, loss=0.24997849762439728
DEBUG - subject 52, trial 5, x=e, y=a, y_pred=b, loss=0.25023120641708374
DEBUG - subject 52, trial 6, x=a, y=K, y_pred=W, loss=0.24916476011276245
DEBUG - subject 52, trial 7, x=K, y=d, y_pred=0, loss=0.25108882784843445
DEBUG - subject 52, trial 8, x=d, y=c, y_pred=b, loss=0.25033122301101685
DEBUG - subject 52, trial 9, x=c, y=0, y_pred=d, loss=0.2501128911972046
DEBUG - subject 52, trial 10, x=0, y=f, y_pred=ù, loss=0.2506328523159027
DEBUG - subject 52, trial 11, x=f, y=c, y_pred=H, loss=0.24887384474277496
DEBUG - subject 52, trial 12, x=c, y=J, y_pred=d, loss=0.2498432993888855
DEBUG - subject 52, trial 13, x=J, y=f, 

 88%|███████████████████████████████████████████████████████████████████████████████████████▍           | 53/60 [01:39<00:12,  1.83s/it]

DEBUG - subject 53, trial 0, x=b, y=M, y_pred=U, loss=0.24929888546466827
DEBUG - subject 53, trial 1, x=M, y=e, y_pred=b, loss=0.24942699074745178
DEBUG - subject 53, trial 2, x=e, y=c, y_pred=ù, loss=0.24900205433368683
DEBUG - subject 53, trial 3, x=c, y=Y, y_pred=U, loss=0.2503168284893036
DEBUG - subject 53, trial 4, x=Y, y=f, y_pred=0, loss=0.25087353587150574
DEBUG - subject 53, trial 5, x=f, y=a, y_pred=ù, loss=0.2500057816505432
DEBUG - subject 53, trial 6, x=a, y=P, y_pred=Y, loss=0.2509736120700836
DEBUG - subject 53, trial 7, x=P, y=d, y_pred=U, loss=0.24970240890979767
DEBUG - subject 53, trial 8, x=d, y=c, y_pred=T, loss=0.2491890788078308
DEBUG - subject 53, trial 9, x=c, y=0, y_pred=U, loss=0.2501475214958191
DEBUG - subject 53, trial 10, x=0, y=f, y_pred=W, loss=0.2527675926685333
DEBUG - subject 53, trial 11, x=f, y=c, y_pred=ù, loss=0.25060752034187317
DEBUG - subject 53, trial 12, x=c, y=P, y_pred=P, loss=0.2499290406703949
DEBUG - subject 53, trial 13, x=P, y=f, y_